# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Tumour-Board Briefing

### Scenario

You are the visualization engineer in the **3-D Imaging Lab at City General Hospital**. Tomorrow morning the
neuro-oncology **tumour board** (a neurosurgeon, a radiologist and an oncologist) will review a patient's head CT
(`unit3_ppt3_head_ct_phantom.csv`). The lab head forwards you four requests:

1. **IT department:** *"The new scanner protocol is 512 × 512 × 300 at 0.5 × 0.5 × 1.0 mm. How much storage per scan,
   and how many samples does one volume-rendered frame cost?"*
2. **Radiologist:** *"Show me the lesion in all three planes, at its centre, with the anatomy undistorted."*
3. **Neurosurgeon:** *"I need the tumour volume in mL — and I want to know how much that number depends on the
   threshold you picked."*
4. **Oncologist:** *"Your last 3-D render showed a beautiful skull and nothing else. I need to see the tumour and its
   relation to the ventricles."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your
own words. The ground-truth `labels` array may be used **only to evaluate** your results, never to find the tumour.

## Part 0 — Setup (given)

Run this cell as-is. It loads the volume `vol[z, y, x]`, the evaluation-only `labels`, the voxel spacing, and two
helpers: `make_tf(points)` builds a transfer function from `(hu, (r, g, b), alpha)` control points, and
`render(volume, tf, axis)` ray-casts the volume with front-to-back compositing (resampled at 2× density).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ct = pd.read_csv("unit3_ppt3_head_ct_phantom.csv")
NZ, NY, NX = ct.z.max() + 1, ct.y.max() + 1, ct.x.max() + 1
vol = ct.intensity_hu.values.reshape(NZ, NY, NX)        # V[z, y, x]  (z = feet->head, y = front->back, x = left->right)
labels = ct.label.values.reshape(NZ, NY, NX)            # ground truth: 0 air, 1 scalp, 2 bone, 3 brain, 4 CSF, 5 tumour
SPACING = (4.0, 3.0, 3.0)                               # voxel size in mm along (z, y, x)
VOXEL_MM3 = np.prod(SPACING)
ASPECT = SPACING[0] / SPACING[2]                        # use when z is on a screen axis

def make_tf(points):
    # points: list of (hu, (r, g, b), alpha) control points -> transfer function T(s) -> (C, alpha)
    hu = np.array([p[0] for p in points], float)
    rgb = np.array([p[1] for p in points], float)
    alpha = np.array([p[2] for p in points], float)
    def tf(s):
        c = np.stack([np.interp(s, hu, rgb[:, k]) for k in range(3)], axis=-1)
        return c, np.interp(s, hu, alpha)
    return tf

def render(volume, tf, axis=1, upsample=2):
    # Orthographic ray casting with front-to-back compositing; rays travel along `axis`.
    # upsample=2 resamples the volume trilinearly at twice the density (smoother, fewer aliasing artifacts).
    v = ndimage.zoom(volume.astype(float), upsample, order=1) if upsample > 1 else volume
    v = np.moveaxis(v, axis, 0)
    C = np.zeros(v.shape[1:] + (3,)); A = np.zeros(v.shape[1:])
    for sample in v:                                    # one sample per voxel step along every ray at once
        c, a = tf(sample)
        C += ((1 - A) * a)[..., None] * c
        A += (1 - A) * a
    return C, A

print(f"vol {vol.shape} (z, y, x), voxel {SPACING} mm, {VOXEL_MM3:.0f} mm^3 per voxel")

## Task 1 — The IT Department's Numbers (Request #1)

**Real-world usage:** PACS storage and GPU sizing for 3-D workstations are budgeted from exactly these figures.

**Why this technique:** Voxel count = Nx·Ny·Nz (Slides 7, 12); each ray samples the volume at successive positions
(Slide 22), so one frame costs ≈ rays × samples per ray; early ray termination skips samples once α_acc → 1
(Slide 27).

**TODO:**
1. Number of voxels in a 512 × 512 × 300 scan, and its size in MB at 16 bits per voxel.
2. Physical size of the scanned region in mm, and the volume of one voxel in mm³ (0.5 × 0.5 × 1.0 mm).
3. Samples needed for ONE front-view rendered frame (one ray per x–z pixel, one sample per voxel step in y), and
   for a 360° rotation video of 120 frames.
4. If early ray termination stops rays after 35% of their samples on average, how many samples does the 120-frame
   video need instead?

In [ ]:
n_voxels = 512 * 512 * 300
scan_mb = n_voxels * 2 / 1024**2
extent_mm = (512 * 0.5, 512 * 0.5, 300 * 1.0)
voxel_mm3 = 0.5 * 0.5 * 1.0
samples_per_frame = 512 * 300 * 512
samples_video = samples_per_frame * 120
samples_video_ert = samples_video * (1 - 0.35)

print(f"{n_voxels:,} voxels = {scan_mb:.1f} MB | extent {extent_mm} mm | voxel {voxel_mm3} mm^3")
print(f"samples/frame {samples_per_frame:,} | 120 frames {samples_video:,} | with early termination {samples_video_ert:,.0f}")


**Reflection:** Ignoring anisotropic spacing would make coronal and sagittal anatomy appear stretched or compressed along the z direction, so distances and shapes would be wrong. It would also misplace the cross-sectional geometry relative to the other planes, producing incorrect measurements and misleading spatial relationships.


## Task 2 — Find the Lesion and Show It in Three Planes (Request #2)

**Real-world usage:** Every radiology workstation opens on a 3-plane MPR with linked cross-hairs.

**Why this technique:** **Orthogonal slices / MPR** give complementary views of the same structure (Slides 15–17);
a **window/level** setting is the slice-display transfer function; the correct **voxel aspect ratio** keeps anatomy
undistorted (Slide 9).

**TODO:**
1. Find the lesion *without* `labels`: take voxels with 55 < HU < 90 that lie **inside the brain** — use
   `inside = ndimage.binary_fill_holes(vol > 300) & ~(vol > 300)` and erode it 3 times
   (`ndimage.binary_erosion(..., iterations=3)`) to avoid the partial-volume shell under the skull.
2. Keep the largest connected component (`ndimage.label`) and compute its centroid (z, y, x) as integers.
3. Plot axial, coronal and sagittal slices through that centroid with cross-hairs, a brain window (W = 80, L = 40),
   and `aspect=ASPECT` for the two views that contain z.
4. Evaluation only: print how far (in mm) your centroid is from the centroid of `labels == 5`.

In [ ]:
def window(a, W, L):
    return np.clip((a - (L - W / 2)) / W, 0, 1)

inside = ndimage.binary_fill_holes(vol > 300) & ~(vol > 300)
inside = ndimage.binary_erosion(inside, iterations=3)
candidates = (vol > 55) & (vol < 90) & inside
cc, n_cc = ndimage.label(candidates)
sizes = np.bincount(cc.ravel()); sizes[0] = 0
lesion = cc == sizes.argmax()
cz, cy, cx = np.rint(ndimage.center_of_mass(lesion)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
views = [(vol[cz], "Axial (z)", cy, cx, None), (vol[:, cy, :], "Coronal (y)", cz, cx, ASPECT), (vol[:, :, cx], "Sagittal (x)", cz, cy, ASPECT)]
for ax, (sl, title, hy, hx, aspect) in zip(axes, views):
    ax.imshow(window(sl, 80, 40), cmap="gray", aspect=aspect if aspect is not None else "auto")
    ax.axhline(hy, color="cyan", lw=0.7); ax.axvline(hx, color="cyan", lw=0.7); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()
truth_centroid = np.array(ndimage.center_of_mass(labels == 5))
distance_mm = np.linalg.norm((np.array([cz, cy, cx]) - truth_centroid) * np.array(SPACING))
print(f"Centroid voxel {(cz, cy, cx)}; distance from evaluation centroid: {distance_mm:.2f} mm")


**Reflection:** Erosion removes the thin partial-volume shell where a voxel mixes skull, CSF and brain, and therefore may fall into the tumour-like HU interval even though it is not tumour. Because tissue intensity ranges overlap, a threshold alone cannot separate those materials reliably; erosion adds a spatial interior constraint before connected-component selection.


## Task 3 — Tumour Volume and Its Threshold Sensitivity (Request #3)

**Real-world usage:** Tumour volume drives treatment decisions and response assessment, so its uncertainty must be
reported.

**Why this technique:** An **isosurface/segmentation at threshold c** defines the structure — and *the chosen c
strongly influences the resulting geometry* (Slides 19–20, 31–32).

**TODO:**
1. For c in `[45, 50, 55, 60, 65, 70]`, grow the region `vol > c` from your lesion centre (connected component
   containing `(cz, cy, cx)`).
2. For each c, record the voxel count, the volume in mL (`voxels * VOXEL_MM3 / 1000`), and — evaluation only — the
   Dice overlap with `labels == 5`.
3. Plot volume (mL) vs. threshold, and mark the threshold with the best Dice.
4. Print the range of volumes you would report to the surgeon.

In [ ]:
truth = labels == 5
rows = []
for c in [45, 50, 55, 60, 65, 70]:
    threshold_cc, _ = ndimage.label(vol > c)
    component_id = threshold_cc[cz, cy, cx]
    segmentation = threshold_cc == component_id
    n = int(segmentation.sum())
    dice = 2 * np.logical_and(segmentation, truth).sum() / (segmentation.sum() + truth.sum())
    rows.append({"threshold": c, "voxels": n, "volume_ml": n * VOXEL_MM3 / 1000, "dice": dice})

table = pd.DataFrame(rows)
display(table)
best_threshold = table.loc[table.dice.idxmax(), "threshold"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(table.threshold, table.volume_ml, "o-"); ax.axvline(best_threshold, color="red", ls="--", label=f"best Dice: {best_threshold:g}")
ax.set(xlabel="HU threshold", ylabel="tumour volume (mL)", title="Tumour volume sensitivity to threshold"); ax.legend()
plt.tight_layout(); plt.show()
print(f"Reportable threshold-sensitivity range: {table.volume_ml.min():.2f}?{table.volume_ml.max():.2f} mL")


**Reflection:** I would choose a threshold using radiologist review, the MPR appearance, and stability of the segmentation across nearby thresholds, then document that rationale with the scan protocol. I would show the central estimate together with the full sensitivity range as an uncertainty band or error bars and overlay the segmentation at low, chosen and high thresholds, rather than presenting one exact volume.


## Task 4 — A 3-D Render That Shows the Tumour (Request #4)

**Real-world usage:** Pre-operative 3-D views show a lesion in context with critical structures (here the ventricles).

**Why this technique:** **Direct volume rendering** with a designed **transfer function** — colour to *identify*
materials, opacity to *reveal or suppress* them (Slides 21–27); occlusion is the main enemy (Slide 28).

**TODO:**
1. Render the given `bone_tf` (front view, `axis=1`) — this is the "beautiful skull" the oncologist complained about.
2. Design your own `tumour_tf` with `make_tf([...])` so that: air and soft tissue are fully transparent, bone is
   *almost* transparent (α ≤ 0.01, still gives context), CSF (≈ 5–20 HU) is blue and faint, and the tumour range
   is red. Remember that opacity **accumulates over every sample along a ray**: a thick structure with α ≈ 0.1
   per sample becomes nearly opaque, while a thin one stays faint — use that to your advantage.
3. Render your TF from the front (`axis=1`) **and** from above (`axis=0`), side by side with the bone render.
4. Give the figure a conclusion-style title.

In [ ]:
bone_tf = make_tf([(-1000, (0, 0, 0), 0), (200, (0, 0, 0), 0), (400, (1, 0.95, 0.85), 0.3), (1500, (1, 1, 1), 0.6)])

tumour_tf = make_tf([(-1000, (0, 0, 0), 0), (4, (0, 0, 0), 0), (20, (0.1, 0.3, 1.0), 0.015), (45, (0.1, 0.3, 1.0), 0.015), (55, (1.0, 0.0, 0.0), 0.12), (90, (1.0, 0.0, 0.0), 0.12), (200, (0, 0, 0), 0.005), (400, (0, 0, 0), 0.005), (1500, (0, 0, 0), 0)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, (rgb, alpha), title in zip(axes, [render(vol, bone_tf, axis=1), render(vol, tumour_tf, axis=1), render(vol, tumour_tf, axis=0)], ["Bone transfer function", "Tumour context: front", "Tumour context: above"]):
    ax.imshow(np.clip(rgb, 0, 1)); ax.set_title(title); ax.axis("off")
fig.suptitle("Tumour visibility improves when opacity suppresses the skull")
plt.tight_layout(); plt.show()


**Reflection:** The faint red rim is caused by the **partial-volume effect** and overlapping material intensities, which make a boundary voxel look tumour-like even when it contains skull/soft-tissue mixture. A segmentation or connected-component mask (or an isosurface with spatial cleanup) can remove that rim by enforcing structure and connectivity. A transfer function only maps intensity to colour/opacity; it cannot know whether the same HU value belongs to tumour or a boundary without spatial information.


**Your final note here:** The lesion is a small central brain lesion near voxel (z=28, y=40, x=41), with threshold-based volumes spanning 6.12?19.26 mL across the tested range. I would open with the three-plane MPR because orthogonal slicing exposes internal structure without the occlusion that can hide a tumour in a surface-like 3-D render. I would then show the segmentation overlay and report a chosen threshold together with the sensitivity range, since segmentation enables measurement but threshold choice changes the geometry. The 3-D render is useful for context, but its transfer function can confuse overlapping intensities and cannot by itself resolve partial-volume boundaries.
